# INCLUDE Pivoting and Category Dominance

Clean execution notebook for the INCLUDE paper figures. Result loading lives in `analysis.py`; plot construction lives in `vis.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import (
    DOMAIN_COMPARISON_DATA_SOURCES,
    INCLUDE_DOMAIN_DATA_SOURCE,
    OPENENDED_METHOD_ORDER,
    MAIN_PAPER_SYNTHETIC_MODELS,
    MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER,
    add_normalized_layers,
    build_pivot_destination_tables,
    load_openended_prompt_metrics,
    load_openended_run_manifest,
)
from vis import (
    plot_domain_comparison_lines,
    save_matplotlib_figure_bundle,
    set_matplotlib_paper_font,
)

for font_path in font_manager.findSystemFonts():
    if "avenir" in font_path.lower():
        font_manager.fontManager.addfont(font_path)

AVENIR_CACHED_FACES = {
    REPO_ROOT / ".cache" / "fonts" / "AvenirNext-Medium.ttf": 5,
    REPO_ROOT / ".cache" / "fonts" / "AvenirNext-DemiBold.ttf": 2,
}
if any(not path.exists() for path in AVENIR_CACHED_FACES):
    from fontTools.ttLib import TTCollection

    avenir_next_ttc = next(
        (Path(path) for path in font_manager.findSystemFonts() if Path(path).name == "Avenir Next.ttc"),
        None,
    )
    if avenir_next_ttc is not None:
        collection = TTCollection(avenir_next_ttc)
        for cache_path, face_index in AVENIR_CACHED_FACES.items():
            if not cache_path.exists():
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                collection.fonts[face_index].save(cache_path)
for cache_path in AVENIR_CACHED_FACES:
    if cache_path.exists():
        font_manager.fontManager.addfont(cache_path)

set_matplotlib_paper_font()

LOG_ROOT = REPO_ROOT / "logs" / "evals"
FIG_DIR = REPO_ROOT / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PAPER_MAIN_FIG_DIR = FIG_DIR / "paper" / "main"
PAPER_APPENDIX_FIG_DIR = FIG_DIR / "paper" / "appendix"

## Load Domain Comparison Runs

Methods are `repr`, `raw-rmax`, and `raw-rtopp`. The manifest keeps one latest run per dataset, model, and method. Missing failed runs remain absent and plot as blank cells.

In [ ]:
manifest = load_openended_run_manifest(
    LOG_ROOT,
    data_sources=DOMAIN_COMPARISON_DATA_SOURCES,
    methods=OPENENDED_METHOD_ORDER,
)

manifest_summary = (
    manifest
    .groupby(["data_source", "method_label"], dropna=False)["display_model_name"]
    .nunique()
    .unstack(fill_value=0)
)
manifest_summary


## Domain Dominance and Pivoting


In [ ]:
# Build the OLMo-inclusive model subset used by the appendix domain figures.
DOMAIN_COMPARISON_MODELS = [
    model for model in MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER
    if model in {*MAIN_PAPER_SYNTHETIC_MODELS, "OLMo-2-1124-7B"}
]
OLMO_DISPLAY_NAME = "OLMo-2-1124-7B"
if OLMO_DISPLAY_NAME not in DOMAIN_COMPARISON_MODELS:
    DOMAIN_COMPARISON_MODELS.append(OLMO_DISPLAY_NAME)

domain_manifest_with_olmo = manifest[
    manifest["display_model_name"].isin(DOMAIN_COMPARISON_MODELS)
    & (
        manifest["display_model_name"].ne(OLMO_DISPLAY_NAME)
        | manifest["revision"].eq("main")
    )
].copy()

olmo_rows = domain_manifest_with_olmo[domain_manifest_with_olmo["display_model_name"].eq(OLMO_DISPLAY_NAME)]
if olmo_rows.empty:
    print("No OLMo-2 main-checkpoint domain-line runs found in the selected manifest.")
else:
    print("OLMo-2 main-checkpoint domain-line runs:")
    print(
        olmo_rows[["data_source", "method_label", "model_name", "revision", "exp_id"]]
        .sort_values(["data_source", "method_label", "exp_id"])
        .to_string(index=False)
    )

domain_comparison_df = load_openended_prompt_metrics(domain_manifest_with_olmo)
domain_comparison_df = add_normalized_layers(domain_comparison_df)

fig = plot_domain_comparison_lines(
    domain_comparison_df,
    metric="dominance",
    data_sources=DOMAIN_COMPARISON_DATA_SOURCES,
    layer_col="layer",
    methods=OPENENDED_METHOD_ORDER,
    models=DOMAIN_COMPARISON_MODELS,
    title=None,
    figsize=(12.5, 12.0),
    panel_label_fontsize=20,
    panel_label_fontweight="bold",
    tick_fontsize=17,
    legend_fontsize=20,
    axis_label_fontsize=21,
    x_axis_label_y=0.138,
    y_axis_label_y=0.57,
)
save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / "10a_domain_dominance")
plt.show()


In [ ]:
# Same OLMo-inclusive model subset as the dominance figure above, now for pivot rate.
fig = plot_domain_comparison_lines(
    domain_comparison_df,
    metric="pivot",
    data_sources=DOMAIN_COMPARISON_DATA_SOURCES,
    layer_col="layer",
    methods=OPENENDED_METHOD_ORDER,
    models=DOMAIN_COMPARISON_MODELS,
    title=None,
    figsize=(12.5, 12.0),
    panel_label_fontsize=20,
    panel_label_fontweight="bold",
    tick_fontsize=17,
    legend_fontsize=20,
    axis_label_fontsize=21,
    x_axis_label_y=0.138,
    y_axis_label_y=0.57,
)
save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / "10b_domain_pivot")
plt.show()


## Pivot Destinations on INCLUDE

Rows are Repr-GMM and raw top-p, and columns are models, including OLMo-2. Each panel shows overall dominance together with English-pivot, other-language-pivot, and dashed total-pivot rates. English task prompts are excluded.

In [ ]:
# Destination grid: rows are estimators, columns are models, with OLMo-2 main added.
PIVOT_DESTINATION_MODELS = ["Llama-2-7B", "Aya-23-8B", "Apertus-8B"]
PIVOT_DESTINATION_METHODS = ["repr", "raw-rtopp"]
PIVOT_DESTINATION_MODELS_WITH_OLMO = [
    model for model in MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER
    if model in {*PIVOT_DESTINATION_MODELS, OLMO_DISPLAY_NAME}
]

olmo_event_frames = []
olmo_rate_frames = []
for method in PIVOT_DESTINATION_METHODS:
    method_events, method_rates = build_pivot_destination_tables(
        domain_comparison_df,
        data_source=INCLUDE_DOMAIN_DATA_SOURCE,
        method=method,
        models=PIVOT_DESTINATION_MODELS_WITH_OLMO,
        task_lang_col="prompt_lang",
        exclude_english_prompts=True,
    )
    olmo_event_frames.append(method_events)
    olmo_rate_frames.append(method_rates)

pivot_destination_events_with_olmo = pd.concat(olmo_event_frames, ignore_index=True)
pivot_destination_rates_with_olmo = pd.concat(olmo_rate_frames, ignore_index=True)
assert not pivot_destination_events_with_olmo["prompt_lang"].eq("en").any()
olmo_decomposition_error = (
    pivot_destination_rates_with_olmo["english_pivot"]
    + pivot_destination_rates_with_olmo["other_pivot"]
    - pivot_destination_rates_with_olmo["total_pivot"]
).abs().max()
assert olmo_decomposition_error < 1e-12

destination_styles = [
    ("dominance", "Dominance", "#7570b3", "-"),
    ("english_pivot", "English pivot", "#d95f02", "-"),
    ("other_pivot", "Other-language pivot", "#1b9e77", "-"),
    ("total_pivot", "Total pivot", "#4d4d4d", "--"),
]
destination_method_labels = {
    "repr": "Repr-GMM",
    "raw-rtopp": "Raw LogitLens Top-p",
}

fig, axes = plt.subplots(
    len(PIVOT_DESTINATION_METHODS),
    len(PIVOT_DESTINATION_MODELS_WITH_OLMO),
    figsize=(13.2, 5.0),
    squeeze=False,
    sharex=True,
    sharey=True,
    gridspec_kw={"hspace": 0.24, "wspace": 0.12},
)
for row_idx, method in enumerate(PIVOT_DESTINATION_METHODS):
    for col_idx, model in enumerate(PIVOT_DESTINATION_MODELS_WITH_OLMO):
        ax = axes[row_idx, col_idx]
        panel = pivot_destination_rates_with_olmo[
            pivot_destination_rates_with_olmo["display_model_name"].eq(model)
            & pivot_destination_rates_with_olmo["method_label"].eq(method)
        ].sort_values("layer")
        if panel.empty:
            ax.text(
                0.5,
                0.5,
                f"{destination_method_labels.get(method, method)} results not available yet\nfor {model}",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=14,
                color="#777777",
            )
        else:
            for value_col, label, color, linestyle in destination_styles:
                ax.plot(
                    panel["layer"],
                    panel[value_col],
                    color=color,
                    linestyle=linestyle,
                    linewidth=2.0 if value_col != "total_pivot" else 1.6,
                    label=label,
                )
        ax.set_ylim(0, 1)
        ax.grid(True, linewidth=0.4, alpha=0.25)
        if row_idx == 0:
            ax.set_title(model, fontsize=16, fontfamily="Avenir Next", fontweight=600)
        if col_idx == 0:
            ax.set_ylabel(
                destination_method_labels.get(method, method),
                fontsize=16,
                fontfamily="Avenir Next",
                fontweight=600,
                labelpad=11,
            )
        ax.tick_params(labelsize=14)
        ax.tick_params(axis="x", labelbottom=True)

destination_handles = [
    plt.Line2D([0], [0], color=color, linestyle=linestyle, lw=3.0, label=label)
    for _, label, color, linestyle in destination_styles
]
fig.legend(
    handles=destination_handles,
    loc="lower center",
    bbox_to_anchor=(0.56, 0.005),
    ncol=len(destination_handles),
    frameon=False,
    fontsize=16,
)
fig.supxlabel("Layer", fontsize=18, x=0.56, y=0.105)
fig.supylabel("Pivot Rate / Dominance", fontsize=18, x=0.035)
fig.subplots_adjust(left=0.13, right=0.99, bottom=0.225, top=0.92)
save_matplotlib_figure_bundle(
    fig,
    PAPER_MAIN_FIG_DIR / "05_include_pivot_cat_dominance",
)
plt.show()
